# Outliers, Inconsistencies, and Data Quality

**Data Wrangling — Fall 2026** · Companion notebook to the lecture slides

---

## How to use this notebook

Run the cells in order. Each of the three examples from the slides is reproduced
here in full, so you can change the numbers and watch the conclusions change.

| Section | What it covers |
|---|---|
| 0 | Setup and a reusable outlier-detection toolkit |
| 1 | **Example 1** — The Bison Half-Marathon (IQR rule) |
| 2 | **Example 2** — The \$1 Apartment (why z-scores fail) |
| 3 | **Example 3** — The Ice Cream Truck (contextual outliers) |
| 4 | Handling strategies: fix, remove, cap, transform, keep |
| 5 | Fixing inconsistencies |
| 6 | The data quality report |
| 7 | **Practice problems** (solutions at the bottom) |

> **The one rule for this notebook:** a flagged value is a *question*, not a
> verdict. Every time something gets flagged, ask what happened before you ask
> how to delete it.

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("pandas", pd.__version__, "| numpy", np.__version__)

### 0.1 The detection toolkit

Three detectors, one interface. Keep this cell — you will reuse these functions
all semester.

In [ ]:
def z_scores(x):
    """Classic z-score. Uses the SAMPLE sd (ddof=1), like pandas .std()."""
    x = pd.Series(x).astype(float)
    return (x - x.mean()) / x.std(ddof=1)


def iqr_fences(x, k=1.5):
    """Tukey's fences. k=1.5 is 'outlier', k=3.0 is 'far out'."""
    x = pd.Series(x).astype(float)
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def modified_z(x):
    """Robust z-score built on the median and MAD.

    Returns NaN everywhere if MAD == 0 (happens when >50% of values are equal).
    """
    x = pd.Series(x).astype(float)
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0:
        print("  [warning] MAD is 0 - more than half the values are identical.")
        return pd.Series(np.nan, index=x.index)
    return 0.6745 * (x - med) / mad


def detect_all(x, z_cut=3.0, k=1.5, m_cut=3.5):
    """Run all three detectors and return a tidy comparison table."""
    x = pd.Series(x).astype(float)
    lo, hi = iqr_fences(x, k)
    z = z_scores(x)
    m = modified_z(x)
    return pd.DataFrame({
        "value":      x,
        "z":          z.round(3),
        "z_flag":     z.abs() > z_cut,
        "iqr_flag":   (x < lo) | (x > hi),
        "mod_z":      m.round(2),
        "mod_z_flag": m.abs() > m_cut,
    })

### 0.2 A sanity check on the toolkit

Before trusting a function, test it on data where you already know the answer.

In [ ]:
# 100 well-behaved values plus one obvious intruder
test = pd.Series(list(np.linspace(10, 20, 100)) + [500.0])

out = detect_all(test)
print("rows flagged by each method (out of 101):")
print(out[["z_flag", "iqr_flag", "mod_z_flag"]].sum())
print("\nthe flagged value(s):")
print(out[out["mod_z_flag"]])

---
## 1. Example 1 — The Bison Half-Marathon

Fifteen runners in one age group. One of them apparently finished a half
marathon in **47 minutes**. The world record is about 57 minutes.

In [ ]:
times = pd.Series(
    [92, 95, 97, 98, 101, 103, 104, 106, 108, 110, 112, 115, 118, 121, 47],
    name="minutes", dtype=float,
)
print("n =", len(times))
print("sorted:", sorted(times.astype(int)))

### 1.1 The IQR rule, by hand and in code

On the slides we used the textbook position rule, $Q_1$ at position
$\frac{n+1}{4}$. NumPy interpolates instead. **The numbers differ slightly and
the conclusion is identical** — this is normal, not a bug.

In [ ]:
s = np.sort(times.values)

# --- textbook convention: positions (n+1)/4, (n+1)/2, 3(n+1)/4 ---
n = len(s)
q1_hand = s[int((n + 1) / 4) - 1]
q3_hand = s[int(3 * (n + 1) / 4) - 1]
iqr_hand = q3_hand - q1_hand

# --- numpy convention: linear interpolation ---
q1_np, q3_np = np.percentile(times, [25, 75])
iqr_np = q3_np - q1_np

comparison = pd.DataFrame({
    "by hand": [q1_hand, q3_hand, iqr_hand,
                q1_hand - 1.5 * iqr_hand, q3_hand + 1.5 * iqr_hand],
    "numpy":   [q1_np, q3_np, iqr_np,
                q1_np - 1.5 * iqr_np, q3_np + 1.5 * iqr_np],
}, index=["Q1", "Q3", "IQR", "lower fence", "upper fence"])
print(comparison)

lo, hi = iqr_fences(times)
print(f"\nflagged: {sorted(times[(times < lo) | (times > hi)].tolist())}")

### 1.2 What the outlier does to the summary statistics

Watch the standard deviation, not the mean. The mean shifts by 4 minutes; the
**spread nearly doubles**, and spread is what every significance test divides by.

In [ ]:
clean = times[times > 60]

summary = pd.DataFrame({
    "with 47":    [times.mean(), times.median(), times.std(ddof=1)],
    "without 47": [clean.mean(), clean.median(), clean.std(ddof=1)],
}, index=["mean", "median", "std"]).round(2)
summary["change"] = (summary["with 47"] - summary["without 47"]).round(2)
print(summary)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].boxplot(times, vert=False, widths=0.6)
ax[0].scatter(times, np.ones(len(times)), alpha=0.6, zorder=3)
ax[0].axvline(lo, ls="--", lw=1, color="crimson")
ax[0].axvline(hi, ls="--", lw=1, color="crimson")
ax[0].set_title("All 15 runners (dashed = Tukey fences)")
ax[0].set_xlabel("finishing time (minutes)")
ax[0].set_yticks([])

ax[1].boxplot(clean, vert=False, widths=0.6)
ax[1].scatter(clean, np.ones(len(clean)), alpha=0.6, zorder=3)
ax[1].set_title("After removing the 47")
ax[1].set_xlabel("finishing time (minutes)")
ax[1].set_yticks([])

plt.tight_layout()
plt.show()

### 1.3 The verdict

The statistics **flagged** the value. Domain knowledge **classified** it: no
human has run a half marathon in 47 minutes, so this is an *error*, not a rare
result.

That means the left branch of the decision framework — **investigate and fix**.
Before deleting the row, the race director should check the timing-mat splits
and the bib assignment. A recovered true value always beats a deleted row.

---
## 2. Example 2 — The \$1 Apartment

Twelve monthly rents scraped from a listings site. Two are wrong:

- **\$1** — a placeholder a landlord typed to win "lowest price" sorts
- **\$45,000** — an annual rent entered into a monthly field

**The question:** does the familiar $|z| > 3$ rule catch them?

In [ ]:
rents = pd.Series(
    [1800, 1950, 2000, 2100, 2150, 2200, 2250, 2300, 2400, 2500, 1, 45000],
    name="rent", dtype=float,
)
print(f"n = {len(rents)}")
print(f"naive mean = ${rents.mean():,.2f}")
print(f"...but 10 of 12 apartments rent for $2,500 or less.")

### 2.1 The z-score fails twice

In [ ]:
result = detect_all(rents).sort_values("value")
print(result.to_string())

Two separate failures in that table:

**Failure 1 — the \$1 is invisible.** Its z-score is $-0.446$, which is *more
ordinary-looking* than a real \$2,500 rent at $-0.246$. The \$45,000 dragged the
mean up to \$5,554 and inflated the SD to \$12,439; against a yardstick that
large, the distance down to \$1 is nothing. **This is called masking: the big
outlier hides the small one.**

**Failure 2 — the \$45,000 barely qualifies.** There is a hard ceiling on how
large a z-score can be in a sample of size $n$:

$$|z|_{\max} = \frac{n-1}{\sqrt{n}}$$

In [ ]:
n = len(rents)
ceiling = (n - 1) / np.sqrt(n)
z_big = z_scores(rents).max()

print(f"n = {n}")
print(f"largest z-score mathematically possible : {ceiling:.4f}")
print(f"z-score actually achieved by the $45,000: {z_big:.4f}")
print(f"that is {100 * z_big / ceiling:.1f}% of the theoretical maximum.")
print()
print(f"So the ONLY window in which the |z|>3 rule can ever fire for n=12")
print(f"runs from 3.000 to {ceiling:.3f} - a target {ceiling - 3:.3f} wide.")
print(f"Move the cutoff to 3.2 and NOTHING in a 12-row sample can be flagged,")
print(f"no matter how broken it is.")

print("\nceiling by sample size:")
for k in [5, 10, 12, 20, 50, 100]:
    fires = "never" if (k - 1) / np.sqrt(k) < 3 else "possible"
    print(f"  n={k:>4}  |z|max = {(k - 1) / np.sqrt(k):>5.2f}   "
          f"|z|>3 rule: {fires}")

> **Consequence.** With $n \le 10$, "no value exceeded $|z| > 3$" tells you
> **nothing about your data**. It is a fact about arithmetic. And $n \le 10$ is
> the size of a typical class project.

### 2.2 The modified z-score succeeds

Same formula shape, robust ingredients. Let's build MAD step by step so the
phrase "median of the absolute deviations from the median" stops being
confusing.

In [ ]:
med = rents.median()
deviations = (rents - med).abs()
mad = deviations.median()

print(f"step 1  median of the rents        = {med:,.1f}")
print(f"step 2  distances from the median  = "
      f"{sorted(deviations.astype(int).tolist())}")
print(f"step 3  median of those distances  = {mad:,.1f}   <- this is MAD")

print("\nNotice: the $45,000 contributed the distance 42,825 to that list,")
print("and taking a MEDIAN threw it away. The outlier got no vote in setting")
print("the yardstick. Compare with the SD, where it set the yardstick almost")
print("single-handedly.")

In [ ]:
m = modified_z(rents)
verdict = pd.DataFrame({
    "rent": rents,
    "z": z_scores(rents).round(3),
    "z says": np.where(z_scores(rents).abs() > 3, "FLAG", "fine"),
    "mod_z": m.round(2),
    "mod_z says": np.where(m.abs() > 3.5, "FLAG", "fine"),
}).sort_values("rent")
print(verdict.to_string(index=False))

lo, hi = iqr_fences(rents)
print(f"\nIQR fences: ({lo:,.2f}, {hi:,.2f})  -> also flags both")

good = rents[(rents > lo) & (rents < hi)]
print(f"\nnaive mean     : ${rents.mean():,.2f}")
print(f"corrected mean : ${good.mean():,.2f}   (n={len(good)})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].bar(range(len(rents)), rents.values, color="steelblue")
ax[0].set_yscale("log")
ax[0].set_title("All 12 listings (log scale — note the two spikes)")
ax[0].set_ylabel("monthly rent ($, log)")
ax[0].set_xlabel("listing index")

ax[1].bar(range(len(good)), good.values, color="seagreen")
ax[1].axhline(good.mean(), ls="--", color="crimson",
              label=f"mean = ${good.mean():,.0f}")
ax[1].set_title("After robust screening")
ax[1].set_ylabel("monthly rent ($)")
ax[1].set_xlabel("listing index")
ax[1].legend()

plt.tight_layout()
plt.show()

### 2.3 What you write in the report

> *Mean rent near campus is **\$2,165** (n = 10). Two listings were excluded as
> data errors: one \$1 placeholder and one \$45,000 figure that appears to be an
> annual rent entered in a monthly field. Including them, the mean would be
> \$5,554.*

Three sentences. That is a reproducible analysis: what you removed, why, and
what difference it made.

---
## 3. Example 3 — The Ice Cream Truck

An ice cream truck parks outside the library. For 20 days the owner logs the
temperature and the units sold. She wants to know how much to stock tomorrow.

**Two different problems are hiding in this table.** One is an inconsistency.
One is a contextual outlier that *no single-column method can see*.

In [ ]:
ice = pd.DataFrame({
    "day": [f"D{i:02d}" for i in range(1, 21)],
    "temp_c": [15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 75.2,
               25.0, 26.0, 27.0, 82.4, 29.0, 30.0, 31.0, 32.0, 20.0, 24.0],
    "sales":  [210, 233, 241, 248, 274, 282, 319, 360, 345, 361,
               399, 414, 428, 427, 461, 493, 474, 508, 450, 349],
})
ice

### 3.1 Problem 1 — the unit inconsistency

You do not need statistics here. You need to know that Washington DC has never
been 75 °C. **A validity rule beats a statistical detector whenever you know the
domain.**

In [ ]:
RULES = {"temp_c": (-20, 45), "sales": (0, 2000)}

def check_ranges(df, rules):
    bad = []
    for col, (lo, hi) in rules.items():
        v = df[(df[col] < lo) | (df[col] > hi)]
        for _, r in v.iterrows():
            bad.append({"row": r.get("day", _), "column": col,
                        "value": r[col], "allowed": f"[{lo}, {hi}]"})
    return pd.DataFrame(bad)

violations = check_ranges(ice, RULES)
print(violations.to_string(index=False) if len(violations)
      else "no range violations")

In [ ]:
# Those are Fahrenheit. Convert them - do NOT delete the rows.
ice_fixed = ice.copy()
mask = ice_fixed["temp_c"] > 45
ice_fixed.loc[mask, "temp_c"] = (ice_fixed.loc[mask, "temp_c"] - 32) * 5 / 9

print("repaired rows:")
print(ice_fixed.loc[mask, ["day", "temp_c", "sales"]].to_string(index=False))
print("\nremaining range violations:", len(check_ranges(ice_fixed, RULES)))

> **Convert, do not delete.** Dropping those two rows would cost 10% of a
> 20-row dataset for no reason. The observations were fine; the *encoding* was
> not.
>
> *(NASA lost the \$327M Mars Climate Orbiter in 1999 to exactly this class of
> bug — pound-force seconds vs. newton-seconds. A range check would have caught
> it. Nobody wrote one.)*

### 3.2 Problem 2 — the outlier every univariate method misses

Day 19: 20 °C, **450 units sold**. Run the full toolkit on the sales column.

In [ ]:
sales_check = detect_all(ice_fixed["sales"])
d19 = sales_check.loc[18]
lo, hi = iqr_fences(ice_fixed["sales"])

print("D19 (20 C, 450 units) under every univariate test:")
print(f"  range check  : 0 <= 450 <= 2000            -> not flagged")
print(f"  z-score      : z = {d19['z']:.2f}  (cutoff 3.0)      -> not flagged")
print(f"  IQR fences   : ({lo:.1f}, {hi:.1f}), 450 is inside -> not flagged")
print(f"  mod z-score  : M = {d19['mod_z']:.2f} (cutoff 3.5)      -> not flagged")
print(f"\n  max sales anywhere in the data: {ice_fixed['sales'].max():.0f} units")
print(f"  total rows flagged by ANY method: "
      f"{sales_check[['z_flag','iqr_flag','mod_z_flag']].any(axis=1).sum()}")

**Every tool says the column is clean — and every tool is wrong.**

450 units is unremarkable *for a 29 °C day*. On a 20 °C day the truck sells
about 282. Day 6 was also 20 °C and sold 282 units. The value is only strange
**next to the temperature column**.

If one column cannot see it, stop looking at one column. Model the relationship,
then examine what the model fails to explain.

In [ ]:
# Fit on the ordinary days (exclude D19 so it cannot pull its own prediction)
ordinary = ice_fixed.drop(index=18)
slope, intercept = np.polyfit(ordinary["temp_c"], ordinary["sales"], 1)

ice_fixed["predicted"] = (intercept + slope * ice_fixed["temp_c"]).round(1)
ice_fixed["residual"] = (ice_fixed["sales"] - ice_fixed["predicted"]).round(1)
ice_fixed["resid_z"] = (ice_fixed["residual"]
                        / ice_fixed["residual"].std(ddof=1)).round(2)

print(f"fitted line:  sales = {intercept:.1f} + {slope:.1f} * temp_c\n")
print(ice_fixed[["day", "temp_c", "sales", "predicted",
                 "residual", "resid_z"]].to_string(index=False))

flagged = ice_fixed[ice_fixed["resid_z"].abs() > 3]
print(f"\nflagged on residuals (|resid_z| > 3):")
print(flagged[["day", "temp_c", "sales", "predicted", "resid_z"]]
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))

normal = ice_fixed.drop(index=18)
ax.scatter(normal["temp_c"], normal["sales"], s=55,
           color="steelblue", label="ordinary days", zorder=3)
ax.scatter(ice_fixed.loc[18, "temp_c"], ice_fixed.loc[18, "sales"],
           s=180, color="crimson", marker="*",
           label="D19 — flagged", zorder=4)

xs = np.linspace(14, 33, 50)
ax.plot(xs, intercept + slope * xs, ls="--", color="grey",
        label=f"fit: sales = {intercept:.0f} + {slope:.0f}·temp")

ax.annotate("", xy=(20, 450), xytext=(20, 296),
            arrowprops=dict(arrowstyle="<->", color="crimson", lw=1.6))
ax.text(20.4, 370, "+154 units\nunexplained",
        color="crimson", fontsize=9, va="center")

ax.set_xlabel("temperature (°C)")
ax.set_ylabel("units sold")
ax.set_title("Invisible in one column. Obvious in two.")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 3.3 The verdict — and the most important idea in this notebook

The owner checks her calendar. **Day 19 was Homecoming.** Ten thousand extra
people walked past the truck. The value is *real*.

| | |
|---|---|
| **The wrong move** | Delete D19. The model gets a nicer $R^2$ — and now predicts 296 units for the next big event. The truck sells out by noon and loses a third of its peak revenue. |
| **The right move** | Keep it, and add an `event_day` column. D19 stops being an unexplained outlier and becomes the *most valuable row in the dataset* — the only evidence of what events do to demand. |

> **An outlier you cannot explain is a data quality problem.**
> **An outlier you *can* explain is a missing variable.**
>
> Ask *what is different about this row?* before you ask *how do I remove this
> row?*

In [ ]:
# Encode the explanation instead of deleting the evidence
ice_fixed["event_day"] = False
ice_fixed.loc[18, "event_day"] = True

print(ice_fixed.groupby("event_day")[["sales"]].agg(["count", "mean"]))
print("\nD19 is now documented, not deleted.")

---
## 4. Handling strategies

Five options, ordered by preference. Fixing beats removing; removing is the one
students reach for first and should reach for last.

In [ ]:
demo = pd.Series([12, 14, 15, 15, 16, 17, 18, 19, 21, 22, 95],
                 name="value", dtype=float)

# 1. REMOVE
lo, hi = iqr_fences(demo)
removed = demo[(demo >= lo) & (demo <= hi)]

# 2. CAP / WINSORIZE - clip to the 5th and 95th percentiles
p5, p95 = demo.quantile([0.05, 0.95])
capped = demo.clip(p5, p95)

# 3. TRANSFORM - log compresses the right tail
logged = np.log(demo)

# 4. KEEP + FLAG - the usual correct answer
flagged = pd.DataFrame({
    "value": demo,
    "is_outlier": (demo < lo) | (demo > hi),
})

print(f"original : n={len(demo):2d}  mean={demo.mean():7.2f}  "
      f"max={demo.max():6.1f}")
print(f"removed  : n={len(removed):2d}  mean={removed.mean():7.2f}  "
      f"max={removed.max():6.1f}")
print(f"capped   : n={len(capped):2d}  mean={capped.mean():7.2f}  "
      f"max={capped.max():6.1f}   <- 95 became {capped.max():.1f}")
print(f"logged   : n={len(logged):2d}  mean={logged.mean():7.2f}  "
      f"max={logged.max():6.1f}   (log scale)")
print(f"\nkeep + flag keeps all {len(demo)} rows and adds a column:")
print(flagged[flagged['is_outlier']].to_string(index=False))

⚠️ **The winsorizing trap.** After capping, the maximum of your data is
`95th percentile`, a value that was *never observed*. If you report that
maximum, you are reporting something that did not happen. Never winsorize
silently.

⚠️ **The log trap.** `np.log(0)` is undefined. For count data containing zeros
use `np.log1p`, and remember that every coefficient in a downstream model now
describes a *multiplicative* effect.

### The non-negotiable rule

Whatever you choose, report the headline number **with and without** the change:

> *"Mean rent is \$2,165 excluding two listings flagged as data errors;
> including them it is \$5,554."*

---
## 5. Fixing inconsistencies

Outliers hide in **numbers**. Inconsistencies hide in **encodings** — and no
statistical detector will ever find them.

In [ ]:
messy = pd.DataFrame({
    "order":      ["A100", "A101", "A102", "A103", "A104", "A105", "A103"],
    "city":       ["Washington", "washington", "Silver Spring", "WASHINGTON",
                   "Silver Spring", "Washington DC ", "WASHINGTON"],
    "qty":        [2, 1, 3, 4, 2, 1, 4],
    "unit_price": [19.99, 45.00, 12.50, 19.99, 30.00, 99.99, 19.99],
    "total":      [39.98, 45.00, 37.50, 79.96, 66.00, 99.99, 79.96],
    "order_date": ["2026-04-02"] * 7,
    "ship_date":  ["2026-04-02", "2026-04-02", "2026-04-03", "2026-04-01",
                   "2026-04-04", "2026-04-05", "2026-04-01"],
})
print("naive grouping says there are", messy["city"].nunique(), "cities:")
print(messy["city"].value_counts(), "\n")

In [ ]:
clean = messy.copy()

# --- 1. category variants -----------------------------------------
clean["city"] = clean["city"].str.strip().str.lower()
CITY_MAP = {"washington dc": "washington"}       # domain knowledge, not a rule
clean["city"] = clean["city"].replace(CITY_MAP)
print("after normalisation:", clean["city"].nunique(), "cities")
print(clean["city"].value_counts().to_dict(), "\n")

# --- 2. dates as real dates ---------------------------------------
for c in ["order_date", "ship_date"]:
    clean[c] = pd.to_datetime(clean[c], format="%Y-%m-%d")

# --- 3. cross-field arithmetic ------------------------------------
expected = (clean["qty"] * clean["unit_price"]).round(2)
mismatch = clean[(expected - clean["total"]).abs() > 0.01]
print("rows where qty * unit_price != total:")
print(mismatch[["order", "qty", "unit_price", "total"]].to_string(index=False))
print(f"  -> A104 is off by ${66.00 - 60.00:.2f}. Tax? Shipping? A bug?")
print("     The assertion does not tell you which. It tells you to go ask.\n")

# --- 4. temporal logic --------------------------------------------
impossible = clean[clean["ship_date"] < clean["order_date"]]
print("rows that shipped before they were ordered:")
print(impossible[["order", "order_date", "ship_date"]].to_string(index=False))

# --- 5. duplicate keys --------------------------------------------
dupes = clean[clean.duplicated(subset="order", keep=False)]
print(f"\nduplicate order IDs: {dupes['order'].unique().tolist()}")
print("  -> identical in every field. True duplicate, or a real second order?")
print("     Resolve this by checking the source system, not by guessing.")

### The inconsistency checklist

| Type | Looks like | Fix |
|---|---|---|
| Units | `75.2` and `24.0` in one temperature column | Range rule, then convert |
| Category variants | `"NY"`, `"N.Y."`, `"new york"`, `"New York "` | `.str.strip().str.lower()`, then a mapping dict |
| Date formats | `03/04/2026` — March 4th or April 3rd? | Always pass an explicit `format=`; store ISO |
| Duplicates | Same key twice; same person, two spellings | `.duplicated()` on a key; fuzzy match names |
| Cross-field logic | `qty * price != total`; `ship < order` | Write the rule as an assertion, test every row |
| Sentinels | `999`, `-1`, `"N/A"`, `1900-01-01` | Convert to real `NaN` |

> **Whitespace first.** `"Ana  Reyes"` and `"Ana Reyes"` survive an exact-match
> dedupe. Normalize before you deduplicate, or you will miss duplicates.

---
## 6. The data quality report

Six dimensions. Five of them you can check from inside the file. **Accuracy**
is the hard one — it requires the outside world.

Put this function in a utilities module this week and run it as the first cell
of every notebook for the rest of the semester.

In [ ]:
def quality_report(df, rules=None, key=None):
    """A one-glance quality summary.

    rules : {column: (min, max)} plausible ranges for validity checking
    key   : column that should be unique (a primary key)
    """
    rules = rules or {}
    rep = pd.DataFrame({
        "dtype":     df.dtypes.astype(str),
        "missing_%": (df.isna().mean() * 100).round(1),
        "unique":    df.nunique(),
        "min":       df.min(numeric_only=True),
        "max":       df.max(numeric_only=True),
    })

    invalid = {}
    for col, (lo, hi) in rules.items():
        if col in df.columns:
            invalid[col] = int(((df[col] < lo) | (df[col] > hi)).sum())
    rep["out_of_range"] = pd.Series(invalid)

    print(f"rows: {len(df)}   columns: {df.shape[1]}")
    print(f"fully duplicated rows: {df.duplicated().sum()}")
    if key:
        print(f"duplicate '{key}' values: {df.duplicated(subset=key).sum()}")
    print()
    return rep

In [ ]:
quality_report(ice, rules=RULES)

Reading the report against our three examples:

| Problem | Dimension violated | Response |
|---|---|---|
| \$1 rent | **Validity** | Convert to `NaN` / remove |
| 75.2 °C | **Consistency** (mixed units) | Convert — do not delete |
| Homecoming day | *none — the data is correct* | **Keep and flag** |

Three different dimensions, three different responses. That mapping is the
skill.

---
# 7. Practice Problems

Work these before looking at the solutions. For each one, the deliverable is
**not just the code** — it is the one-sentence justification you would put in a
report.

## Practice Problem 1 — The impossible exam

You are the TA for a course scored out of 100. Here are 14 scores from one
section.

1. Write a **validity rule** for this column. Which values violate it?
2. Compute $Q_1$, $Q_3$, IQR and the Tukey fences **by hand**. Does the IQR rule
   flag the same values your validity rule did? *(Look carefully — it does not.)*
3. The 105 turns out to be a legitimate 95 plus 10 bonus points. The $-5$ was a
   placeholder for "did not sit the exam." **Handle each one**, and say which of
   the five actions from Section 4 you chose and why.
4. Recompute the class mean before and after. Write the one sentence you would
   put in your report.

In [ ]:
scores = pd.Series([72, 68, 91, 85, 77, 80, 88, 74, 95, 83, 105, -5, 79, 86],
                   name="score", dtype=float)

# YOUR WORK HERE

## Practice Problem 2 — The clinic intake sheet

Ten rows from a clinic's patient table. **There are at least seven problems.**
For each one, name the **quality dimension** it violates and state your fix.

*Hint: two of the problems are invisible if you only look one column at a time.*

In [ ]:
clinic = pd.DataFrame({
    "id":   ["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"],
    "name": ["Ana Reyes","Ben Osei","Cara Lin","Dan Meyer","Ana  Reyes",
             "Eve Duarte","Femi Ade","Gus Park","Hana Cruz","Ivan Petrov"],
    "age":  [34, 51, 999, 45, 34, 29, 62, -3, 47, 38],
    "visit_date": ["2026-03-11","2026-03-11","2026-03-12","03/12/2026",
                   "2026-03-11","2026-03-13","2026-13-14","2026-03-14",
                   "2027-03-15","2026-03-15"],
    "bp_systolic": [118, 132, 121, 0, 118, 240, 128, 119, 125, 134],
})
clinic

# YOUR WORK HERE

## Practice Problem 3 — The online store

Use the `messy` DataFrame from Section 5. Every row *looks* fine column by
column, and the dataset is still broken.

1. Group by `city` and count orders. Why is the answer wrong? Write the cleaning
   code.
2. Write a **cross-field assertion** linking `qty`, `unit_price` and `total`.
   Which row fails, and what are three plausible explanations?
3. One row ships *before* it was ordered. Find it.
4. One order ID appears twice. Is it a duplicate row or a real second order?
   **How would you decide?** (There is no statistical answer to this one.)

In [ ]:
# YOUR WORK HERE - messy is defined in Section 5
messy

---
# 8. Solutions

*Try the problems first.*

### Solution 1 — The impossible exam

**The surprise is in part 2:** the validity rule catches both bad values, but
the IQR rule **only catches the $-5$**. With scores spread from 68 to 95, the
upper fence lands above 105.

> **A domain rule can be stronger than a statistical rule.** You know the scale
> maxes at 100. No amount of quartile arithmetic knows that.

The two bad values look similar and need **opposite** treatment: 105 is not an
error at all (the scale allows bonus points — fix your *rule*, not your data),
while $-5$ is a sentinel and must become `NaN`.

In [ ]:
scores = pd.Series([72, 68, 91, 85, 77, 80, 88, 74, 95, 83, 105, -5, 79, 86],
                   name="score", dtype=float)

# --- 1. validity rule ---
VALID = (0, 100)
invalid = scores[(scores < VALID[0]) | (scores > VALID[1])]
print("validity rule 0-100 flags:", invalid.tolist())

# --- 2. IQR rule ---
q1, q3 = scores.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"\nQ1={q1}, Q3={q3}, IQR={iqr}")
print(f"fences = ({lo:.2f}, {hi:.2f})")
print("IQR rule flags:", scores[(scores < lo) | (scores > hi)].tolist())
print(" -> the 105 is NOT flagged: the upper fence sits above it.")

# --- 3. handle each one differently ---
handled = scores.copy()
handled[handled == -5] = np.nan      # sentinel -> missing  (REMOVE from stats)
# 105 stays: it is a real score with bonus credit           (KEEP + FLAG)

# --- 4. before and after ---
print(f"\nmean with everything as-is : {scores.mean():.2f}")
print(f"mean with -5 as missing     : {handled.mean():.2f}")
print("\nReport sentence:")
print('  "One student did not sit the exam and is excluded from the average;')
print('   one score above 100 reflects bonus credit and is retained.')
print(f'   Section mean = {handled.mean():.1f} (n = {handled.notna().sum()})."')

### Solution 2 — The clinic intake sheet

**Eight problems, not seven:**

| # | Row | Problem | Dimension | Fix |
|---|---|---|---|---|
| 1 | P03 | `age = 999` — sentinel | Validity | → `NaN` |
| 2 | P08 | `age = -3` — impossible | Validity | → `NaN`, check source |
| 3 | P04 | `bp = 0` — a living patient has no BP of zero | Validity | → `NaN` |
| 4 | P06 | `bp = 240` — **plausible** (hypertensive crisis) | *none* | **Keep + flag** |
| 5 | P07 | `2026-13-14` — month 13 | Validity | Unparseable → investigate |
| 6 | P09 | visit in **2027** — a future date | Timeliness | Likely a typo for 2026 |
| 7 | P04 | `03/12/2026` while others are ISO | Consistency | Parse explicitly, store ISO |
| 8 | P05 | duplicates P01 (note the **double space**) | Uniqueness | Normalize whitespace, then dedupe |

**#4 is the Homecoming row of this dataset** — statistically an outlier,
medically real. Flag it; do not delete it. It may be the most clinically
important record in the table.

**#8 is why whitespace normalization must come before deduplication** — the
double space defeats an exact-match dedupe.

In [ ]:
c = clinic.copy()

# 1-2. ages
c["age"] = c["age"].where(c["age"].between(0, 120), np.nan)

# 3. impossible blood pressure (0), but keep the plausible 240
c["bp_flag"] = c["bp_systolic"] > 180          # clinically notable, KEEP
c["bp_systolic"] = c["bp_systolic"].where(c["bp_systolic"] > 40, np.nan)

# 5-7. dates: coerce turns unparseable values into NaT instead of crashing
c["visit_date"] = pd.to_datetime(c["visit_date"], errors="coerce",
                                 format="mixed")
future = c["visit_date"] > pd.Timestamp("2026-12-31")

# 8. whitespace FIRST, then deduplicate
c["name"] = c["name"].str.replace(r"\s+", " ", regex=True).str.strip()
dupes = c.duplicated(subset=["name", "age", "visit_date"], keep=False)

print("unparseable dates :", c[c['visit_date'].isna()]['id'].tolist())
print("future dates      :", c[future]['id'].tolist())
print("duplicate patients:", c[dupes]['id'].tolist())
print("flagged BP (keep) :", c[c['bp_flag']]['id'].tolist())
print(f"\nmissing values introduced: {int(c.isna().sum().sum())}")
print("\n(Today's lecture created missing values. Next lecture: what to do")
print(" with them.)")
c

### Solution 3 — The online store

Nothing here is an outlier, and the dataset is still broken.

**Part 1 — four spellings of one city.** `.str.strip().str.lower()` handles
three of them. The fourth (`"Washington DC"`) needs an explicit mapping
dictionary, because no automatic rule knows that "washington dc" and
"washington" are the same market. *Normalization gets you most of the way;
domain knowledge finishes the job.*

**Part 2 — A104.** $2 \times 30.00 = 60.00$, but the total says 66.00. Three
plausible explanations: sales tax, a shipping charge, or a pricing bug. The
assertion does not tell you which — it tells you to go ask somebody. That is the
correct outcome.

**Part 3 — A103** ships 2026-04-01 but was ordered 2026-04-02. Impossible, and
invisible to any check that looks at one column at a time.

**Part 4 — there is no statistical answer.** A103 appears twice, identical in
every field.

- *For a true duplicate:* an order ID should be unique by definition, and every
  field matches exactly.
- *For a real second order:* a customer can genuinely order the same thing
  twice.

**The resolution is procedural.** Check whether `order` is a primary key in the
source system, or look for two payment records. Knowing when the data cannot
answer the question is itself a skill — the right answer here is *"I would go
ask."*

In [ ]:
# the full cleaning pipeline from Section 5, as one function
def clean_orders(df):
    out = df.copy()
    out["city"] = (out["city"].str.strip().str.lower()
                   .replace({"washington dc": "washington"}))
    for col in ["order_date", "ship_date"]:
        out[col] = pd.to_datetime(out[col], format="%Y-%m-%d")
    out["total_ok"] = ((out["qty"] * out["unit_price"]).round(2)
                       - out["total"]).abs() < 0.01
    out["dates_ok"] = out["ship_date"] >= out["order_date"]
    out["dup_key"]  = out.duplicated(subset="order", keep=False)
    return out

audit = clean_orders(messy)
print("cities after cleaning:", audit["city"].value_counts().to_dict())
print("\nrows needing human review:")
print(audit[~audit["total_ok"] | ~audit["dates_ok"] | audit["dup_key"]]
      [["order", "city", "total", "total_ok", "dates_ok", "dup_key"]]
      .to_string(index=False))

---
## What to carry out of this notebook

1. **Investigate, do not delete.** A flagged value is a question, not a verdict.
2. **The z-score is the weakest tool you have.** Zero breakdown point, and for
   $n \le 10$ it is *mathematically incapable* of flagging anything.
3. **Prefer robust methods:** IQR fences and the modified z-score.
4. **Fix inconsistencies before detecting outliers.** Units first, statistics
   second — Example 3 breaks if you reverse the order.
5. **Some outliers are invisible one column at a time.** If nothing looks wrong,
   compare columns.
6. **An outlier you can explain is a missing variable.**
7. **Document every change** and report your result with and without it.

---

### Quick reference

| Method | Built from | Breakdown point | Use when |
|---|---|---|---|
| z-score | mean, SD | **0%** | large $n$, roughly normal, no better option |
| IQR fences | quartiles | 25% | almost always; this is what a boxplot draws |
| Modified z (MAD) | median, MAD | **50%** | small $n$, skewed data, multiple outliers |
| Residuals | a fitted model | depends on the fit | contextual outliers across two or more columns |
| Validity rules | **domain knowledge** | n/a | **always — write these first** |